<a href="https://colab.research.google.com/github/Bhagya028/EDUFYI/blob/main/CNN_transfer_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
import itertools
import keras
from sklearn import metrics
from sklearn.metrics import confusion_matrix
from keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img
from keras.models import Sequential
from keras import optimizers
from keras.preprocessing import image
from keras.layers import Dropout, Flatten, Dense,BatchNormalization,Conv2D,MaxPool2D
from keras import applications
from keras.utils.np_utils import to_categorical
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
%matplotlib inline
import math
import datetime
import time


ImportError: cannot import name 'ImageDataGenerator' from 'keras.preprocessing.image' (/usr/local/lib/python3.12/dist-packages/keras/preprocessing/image/__init__.py)

In [ ]:
img_width, img_height = 128,128
# loading up our datasets

train_data_dir = r'/content/drive/MyDrive/Vani-dev/xray_dataset_covid19/train'
#validation_data_dir = ‘data/validation’
test_data_dir = r'/content/drive/MyDrive/Vani-dev/xray_dataset_covid19/test'

In [ ]:
# number of epochs to train top model
epochs = 7 #this has been changed after multiple model run
# batch size used by flow_from_directory and predict_generator
batch_size = 32

# Image Augmentation

In [ ]:
def get_generator(path):
    generator = datagen.flow_from_directory(
        path,
        target_size=(img_width, img_height),
        batch_size=batch_size,
        class_mode="categorical",
        shuffle=False)

    nb_samples = len(generator.filenames)
    num_classes = len(generator.class_indices)

    predict_size_train = int(math.ceil(nb_samples / batch_size))
    # get the class labels for the training data, in the original order
    labels = generator.classes

    # convert the training labels to categorical vectors
    labels = to_categorical(labels, num_classes=num_classes)
    return (generator,nb_samples,predict_size_train,labels)

In [ ]:
datagen = ImageDataGenerator(rescale=1. / 255)

In [ ]:
generator_train,nb_train_samples,predict_size_train,train_labels = get_generator(train_data_dir)
generator_test,nb_test_samples,predict_size_test,test_labels    = get_generator(test_data_dir)
num_classes = 3
class_labels=list(generator_train.class_indices.keys())

# Apply Pre-trained Model

In [ ]:
def apply_pretrainedModel(pretrained_model,generator,predict_size):
    return pretrained_model.predict_generator(generator, predict_size)

# Apply transfer Learning

In [ ]:
def cnn_after_pretrainedModel(input_shape):
    model = Sequential()
    model.add(Flatten(input_shape=input_shape))
    model.add(Dense(128, activation=keras.layers.LeakyReLU(alpha=0.3)))
    model.add(Dropout(0.5))
    model.add(Dense(64, activation=keras.layers.LeakyReLU(alpha=0.3)))
    model.add(Dropout(0.3))
    model.add(Dense(num_classes, activation='softmax'))
    model.compile(loss='categorical_crossentropy',
       optimizer=optimizers.Adam(lr=1e-4),
       metrics=['acc'])
    return model;

In [ ]:
def transfer_learning(pretrained_model,epochs=25):
    #Appling Pretrained Model to train and test datasets
    train_data = apply_pretrainedModel(pretrained_model,generator_train,predict_size_train)
    test_data  = apply_pretrainedModel(pretrained_model,generator_test,predict_size_test)
    model      = cnn_after_pretrainedModel(train_data.shape[1:])
    history = model.fit(train_data, train_labels,
       epochs=epochs,
       batch_size=batch_size,
       validation_data=(test_data, test_labels))
    (eval_loss, eval_accuracy) = model.evaluate(
        test_data, test_labels, batch_size=batch_size,     verbose=1)
    print("[INFO] accuracy: {:.2f}%".format(eval_accuracy * 100))
    print("[INFO] Loss: {}".format(eval_loss))

    return train_data,test_data,model,history,eval_accuracy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
def plot_graphs(history):
    #Graphing our training and validation
    acc = history.history['acc']
    val_acc = history.history['val_acc']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs = range(len(acc))
    plt.plot(epochs, acc, 'r', label='Training acc')
    plt.plot(epochs, val_acc, 'b', label='Validation acc')
    plt.title('Training and validation accuracy')
    plt.ylabel('accuracy')
    plt.xlabel('epoch')
    plt.legend()
    plt.figure()
    plt.plot(epochs, loss, 'r', label='Training loss')
    plt.plot(epochs, val_loss, 'b', label='Validation loss')
    plt.title('Training and validation loss')
    plt.ylabel('loss')
    plt.xlabel('epoch')
    plt.legend()
    plt.show()

# Build Model  VGG

In [ ]:
#Loading Pre-trained Model
model_name="VGG"
pretrained_model_vgg  = applications.VGG16(include_top=False, weights='imagenet')
train_data,test_data,model_vgg,history_vgg,acc_vgg = transfer_learning(pretrained_model_vgg)
print()

# Visualise VGG training and testing

In [ ]:
plot_graphs(history_vgg)
preds=np.round(model_vgg.predict(test_data),0)

# Performance Metrics

In [ ]:
classify_matrics_vgg=metrics.classification_report(test_labels,preds,target_names=class_labels)
print(classify_matrics_vgg)

## Applying GOOGLE NET

In [ ]:
#Loading Pre-trained Model
model_name="INCEPTION NET"
pretrained_model_googleNet  = applications.InceptionV3(include_top=False, weights='imagenet')
train_data,test_data,model_googleNet,history_googleNet,acc_googleNet = transfer_learning(pretrained_model_googleNet)

In [ ]:
plot_graphs(history_googleNet)
preds=np.round(model_googleNet.predict(test_data),0)

In [ ]:
classify_matrics_googleNet=metrics.classification_report(test_labels,preds,target_names=class_labels)
print(classify_matrics_googleNet)

# ResNet

In [ ]:
#Loading Pre-trained Model
model_name="INCEPTION NET"
pretrained_model_resNet  = applications.ResNet50(include_top=False, weights='imagenet')
train_data,test_data,model_resNet,history_resNet,acc_resNet= transfer_learning(pretrained_model_resNet)

In [ ]:
plot_graphs(history_resNet)
preds=np.round(model_resNet.predict(test_data),0)

In [ ]:
classify_matrics_resNet=metrics.classification_report(test_labels,preds,target_names=class_labels)
print(classify_matrics_resNet)

# Comparing Models

In [ ]:
print("Accuracies of Models")
print("VGG\t\t",acc_vgg)
print("Google Net\t",acc_googleNet)
print("ResNet\t\t",acc_resNet)


# Testing a Single Image

In [ ]:
def read_image(file_path):
   print("[INFO] loading and preprocessing image…")
   image = load_img(file_path, target_size=(img_width, img_height))
   image = img_to_array(image)
   image = np.expand_dims(image, axis=0)
   image /= 255.
   return image

In [ ]:
def test_single_image(pretrained_model,model,path):
  images = read_image(path)
  time.sleep(.5)
  bt_prediction = pretrained_model.predict(images)
  preds = model.predict_proba(bt_prediction)
  for idx, animal, x in zip(range(0,3), class_labels , preds[0]):
   print("ID: {}, Label: {} {}%".format(idx, animal, round(x*100,2) ))
  print('Final Decision:')
  time.sleep(.5)
  for x in range(3):
   print('.'*(x+1))
   time.sleep(.2)
  class_predicted = model.predict_classes(bt_prediction)
  class_dictionary = generator_test.class_indices
  inv_map = {v: k for k, v in class_dictionary.items()}
  print("ID: {}, Label: {}".format(class_predicted[0],  inv_map[class_predicted[0]]))
  return load_img(path)
path = r'/content/drive/MyDrive/Vani-dev/xray_dataset_covid19/test/COVID/SARS-10.1148rg.242035193-g04mr34g0-Fig8c-day10.jpeg'
test_single_image(pretrained_model_googleNet,model_googleNet,path)